<a href="https://colab.research.google.com/github/shagun53/AI-Hand-Gesture-Presentation-Controller/blob/main/Hand_Gesture_AI_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Basic Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TensorFlow
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [4]:
!mkdir -p /content/dataset
!unzip -oq "/content/hand_gesture_dataset.zip" -d "/content/dataset"

In [5]:
df = pd.read_csv("/content/hand_gesture_dataset.csv")
print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (24000, 2)
                               image_path  label
0  /content/dataset/test/test/19/1178.jpg     19
1   /content/dataset/test/test/19/915.jpg     19
2   /content/dataset/test/test/19/992.jpg     19
3  /content/dataset/test/test/19/1173.jpg     19
4   /content/dataset/test/test/19/925.jpg     19


In [6]:
print(df["label"].value_counts())

label
19    1200
0     1200
14    1200
13    1200
6     1200
4     1200
9     1200
5     1200
16    1200
10    1200
17    1200
11    1200
3     1200
2     1200
15    1200
1     1200
18    1200
12    1200
7     1200
8     1200
Name: count, dtype: int64


In [7]:
encoder = LabelEncoder()
df["label_encoded"] = encoder.fit_transform(df["label"])
print(df.head())

                               image_path  label  label_encoded
0  /content/dataset/test/test/19/1178.jpg     19             19
1   /content/dataset/test/test/19/915.jpg     19             19
2   /content/dataset/test/test/19/992.jpg     19             19
3  /content/dataset/test/test/19/1173.jpg     19             19
4   /content/dataset/test/test/19/925.jpg     19             19


In [8]:
label_map = dict(
    zip(
        encoder.classes_,
        encoder.transform(encoder.classes_)
    )
)
print(label_map)

{np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2), np.int64(3): np.int64(3), np.int64(4): np.int64(4), np.int64(5): np.int64(5), np.int64(6): np.int64(6), np.int64(7): np.int64(7), np.int64(8): np.int64(8), np.int64(9): np.int64(9), np.int64(10): np.int64(10), np.int64(11): np.int64(11), np.int64(12): np.int64(12), np.int64(13): np.int64(13), np.int64(14): np.int64(14), np.int64(15): np.int64(15), np.int64(16): np.int64(16), np.int64(17): np.int64(17), np.int64(18): np.int64(18), np.int64(19): np.int64(19)}


In [9]:
df.head()
label_map

{np.int64(0): np.int64(0),
 np.int64(1): np.int64(1),
 np.int64(2): np.int64(2),
 np.int64(3): np.int64(3),
 np.int64(4): np.int64(4),
 np.int64(5): np.int64(5),
 np.int64(6): np.int64(6),
 np.int64(7): np.int64(7),
 np.int64(8): np.int64(8),
 np.int64(9): np.int64(9),
 np.int64(10): np.int64(10),
 np.int64(11): np.int64(11),
 np.int64(12): np.int64(12),
 np.int64(13): np.int64(13),
 np.int64(14): np.int64(14),
 np.int64(15): np.int64(15),
 np.int64(16): np.int64(16),
 np.int64(17): np.int64(17),
 np.int64(18): np.int64(18),
 np.int64(19): np.int64(19)}

In [10]:
print(df.head())
print("\n")
print(df.columns)
print("\n")
print(df["label"].unique())

                               image_path  label  label_encoded
0  /content/dataset/test/test/19/1178.jpg     19             19
1   /content/dataset/test/test/19/915.jpg     19             19
2   /content/dataset/test/test/19/992.jpg     19             19
3  /content/dataset/test/test/19/1173.jpg     19             19
4   /content/dataset/test/test/19/925.jpg     19             19


Index(['image_path', 'label', 'label_encoded'], dtype='object')


[19  0 14 13  6  4  9  5 16 10 17 11  3  2 15  1 18 12  7  8]


In [11]:
print(df.shape)

(24000, 3)


In [12]:
print(df.columns)

Index(['image_path', 'label', 'label_encoded'], dtype='object')


In [13]:
df.head()

,image_path,label,label_encoded
0,/content/dataset/test/test/19/1178.jpg,19,19
1,/content/dataset/test/test/19/915.jpg,19,19
2,/content/dataset/test/test/19/992.jpg,19,19
3,/content/dataset/test/test/19/1173.jpg,19,19
4,/content/dataset/test/test/19/925.jpg,19,19


In [14]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_encoded"]
)

print("Train:", len(train_df))
print("Test :", len(test_df))

Train: 19200
Test : 4800


In [15]:
IMG_SIZE = 224
BATCH_SIZE = 32

In [16]:
import tensorflow as tf
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )
    image = image / 255.0

    return image, label

In [17]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].values,
        train_df["label_encoded"].values
    )
)
train_ds = train_ds.map(load_image)

train_ds = train_ds.shuffle(1000)

train_ds = train_ds.batch(BATCH_SIZE)

train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)
print("Training dataset ready")

Training dataset ready


In [18]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["image_path"].values,
        test_df["label_encoded"].values
    )
)
test_ds = test_ds.map(load_image)

test_ds = test_ds.batch(BATCH_SIZE)

test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)
print("Testing dataset ready")

Testing dataset ready


In [19]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(224,224,3)
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation='relu'
    ),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(
        128,
        (3,3),
        activation='relu'
    ),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(
        128,
        activation='relu'
    ),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(
        20,
        activation='softmax'
    )
])

In [20]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,171,540 (42.62 MB)

 Trainable params: 11,171,540 (42.62 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [23]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 64ms/step - accuracy: 0.9848 - loss: 0.0661 - val_accuracy: 1.0000 - val_loss: 1.6762e-06
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.9993 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 1.6773e-06
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 65ms/step - accuracy: 0.9991 - loss: 0.0039 - val_accuracy: 1.0000 - val_loss: 1.3941e-05
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 62ms/step - accuracy: 0.9996 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 2.1511e-07
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 67ms/step - accuracy: 0.9996 - loss: 0.0013 - val_accuracy: 1.0000 - val_loss: 8.7119e-05
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 1.0000 - loss: 5.3389e-05 - val_accuracy: 1.0000 - val_loss: 1.6561e-06
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 1.0000 - loss: 1.4553e-05 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step 

In [24]:
loss, accuracy = model.evaluate(test_ds)
print("Final Accuracy:", accuracy)

150/150 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 1.0000 - loss: 1.4801e-08
Final Accuracy: 1.0


In [25]:
model.save("/content/gesture_controller.keras")

In [26]:
from google.colab import files
files.download("/content/gesture_controller.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>